# 03 – SQL basics on the portfolio database

Core SQL on my own data: `SELECT`, `WHERE`, `ORDER BY`, `GROUP BY`, `HAVING` and `JOIN`. The database is built by notebook 02 (`data/processed/portfolio.db`).

Each section shows a worked query, followed by an exercise. Write the exercise yourself first; the solutions are at the bottom.

## Setup

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

In [ ]:
DB_PATH = Path("../data/processed/portfolio.db")
assert DB_PATH.exists(), "Run notebook 02 first"


def query(sql):
    """Run a SQL query on the portfolio database and return a DataFrame."""
    with sqlite3.connect(DB_PATH) as con:
        return pd.read_sql(sql, con)

In [ ]:
query("SELECT name FROM sqlite_master WHERE type = 'table'")

## 1. SELECT and LIMIT

`SELECT` chooses columns, `FROM` the table, `LIMIT` the number of rows. SQL keywords in capitals is a convention, not a rule.

In [ ]:
query("""
SELECT ticker, name, pie, weight_in_portfolio
FROM holdings
LIMIT 5
""")

**Exercise 1.** Show `ticker`, `name`, `exchange` and `listing_currency` of the first 10 holdings.

In [ ]:
# Exercise 1


## 2. WHERE: filtering rows

`WHERE` keeps rows that meet a condition. Combine conditions with `AND` / `OR`; text goes between single quotes.

In [ ]:
query("""
SELECT ticker, name, weight_in_portfolio
FROM holdings
WHERE pie = 'Energie' AND weight_in_portfolio > 0.015
""")

**Exercise 2.** Show all ETFs (`asset_type = 'ETF'`) that list in EUR.

In [ ]:
# Exercise 2


## 3. ORDER BY: sorting

`ORDER BY column DESC` sorts from large to small. Combined with `LIMIT` this gives a top-n.

In [ ]:
query("""
SELECT ticker, pie, weight_in_portfolio
FROM holdings
ORDER BY weight_in_portfolio DESC
LIMIT 10
""")

**Exercise 3.** Show the 5 holdings with the highest dividend yield, with `ticker`, `name` and `dividend_yield`.

In [ ]:
# Exercise 3


## 4. GROUP BY and HAVING: aggregating

`GROUP BY` makes one row per group; `SUM`, `COUNT`, `AVG`, `MIN`, `MAX` summarise each group. `WHERE` filters rows **before** grouping, `HAVING` filters groups **after** grouping. This is the SQL version of `groupby` in pandas.

In [ ]:
query("""
SELECT pie, block, COUNT(*) AS n_positions, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY pie, block
ORDER BY block, weight DESC
""")

In [ ]:
query("""
SELECT country, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY country
HAVING SUM(weight_in_portfolio) > 0.05
ORDER BY weight DESC
""")

**Exercise 4.** Total portfolio weight per `listing_currency`, sorted from large to small. Compare the result with `fx_compare` in notebook 01.

In [ ]:
# Exercise 4


## 5. JOIN: combining tables

`JOIN ... ON` links rows of two tables on a shared column, exactly like `merge` in pandas. Short table aliases (`h`, `s`) keep the query readable. `LEFT JOIN` keeps every row of the left table, even without a match.

In [ ]:
query("""
SELECT h.ticker, h.pie, h.weight_in_portfolio, s.yahoo_symbol, s.series_type, s.proxy_symbol
FROM holdings AS h
JOIN symbols AS s ON h.ticker = s.ticker
WHERE s.series_type = 'alt_listing' OR s.proxy_symbol IS NOT NULL
ORDER BY h.weight_in_portfolio DESC
""")

Note `IS NOT NULL`: in SQL, `= NULL` never matches, just as `NaN == NaN` is `False` in pandas.

**Exercise 5.** Total portfolio weight per `series_type` (`listing` / `alt_listing`). Hint: JOIN, then GROUP BY.

In [ ]:
# Exercise 5


## 6. Putting it together on the returns table

How many trading days per ticker come from a proxy, and from which date does each ticker have data? `MIN`, `MAX` and a conditional count with `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`.

In [ ]:
query("""
SELECT ticker,
       MIN(date) AS first_date,
       COUNT(*) AS n_days,
       SUM(CASE WHEN source = 'proxy' THEN 1 ELSE 0 END) AS proxy_days
FROM returns_eur
GROUP BY ticker
HAVING proxy_days > 0 OR MIN(date) > '2019-02-01'
ORDER BY first_date
""")

The worst days of the **current** portfolio (a backtest with today's weights, so it contains hindsight). A JOIN with the weights, a weighted sum per date, sorted ascending. `SUM(h.weight_in_portfolio)` on the same rows shows the coverage of each day: positions without data do not count.

In [ ]:
query("""
SELECT r.date,
       SUM(r.return_total * h.weight_in_portfolio) AS portfolio_return,
       SUM(h.weight_in_portfolio) AS coverage
FROM returns_eur AS r
JOIN holdings AS h ON r.ticker = h.ticker
GROUP BY r.date
ORDER BY portfolio_return ASC
LIMIT 5
""")

**Exercise 6.** The 5 best days of the portfolio since 2022-01-01, with coverage. Hint: the worked query above plus a `WHERE` on the date.

In [ ]:
# Exercise 6


## Solutions

Try each exercise first. Compare afterwards: there is often more than one correct query.

In [ ]:
# Solution 1
query("""
SELECT ticker, name, exchange, listing_currency
FROM holdings
LIMIT 10
""")

In [ ]:
# Solution 2
query("""
SELECT ticker, name, exchange, listing_currency
FROM holdings
WHERE asset_type = 'ETF' AND listing_currency = 'EUR'
""")

In [ ]:
# Solution 3
query("""
SELECT ticker, name, dividend_yield
FROM holdings
ORDER BY dividend_yield DESC
LIMIT 5
""")

In [ ]:
# Solution 4
query("""
SELECT listing_currency, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY listing_currency
ORDER BY weight DESC
""")

In [ ]:
# Solution 5
query("""
SELECT s.series_type, COUNT(*) AS n_positions, SUM(h.weight_in_portfolio) AS weight
FROM holdings AS h
JOIN symbols AS s ON h.ticker = s.ticker
GROUP BY s.series_type
""")

In [ ]:
# Solution 6
query("""
SELECT r.date,
       SUM(r.return_total * h.weight_in_portfolio) AS portfolio_return,
       SUM(h.weight_in_portfolio) AS coverage
FROM returns_eur AS r
JOIN holdings AS h ON r.ticker = h.ticker
WHERE r.date >= '2022-01-01'
GROUP BY r.date
ORDER BY portfolio_return DESC
LIMIT 5
""")